# Características RoCoLe — inventario HSV con máscara COCO

Objetivo: armar un inventario pequeño de histogramas HSV solo dentro de la máscara Leaf de COCO, con el mismo índice y split ya publicados, y guardarlo en un CSV paralelo sin entrenar nada. Es una rama de desarrollo: mismas fotos y partición que el masked RGB/GLCM, pero midiendo solo color en HSV dentro de la hoja. No reescribe el 02 actual.


## Llegar al índice, al COCO y fijar la semilla

Usamos la carpeta featured, el índice partido, el JSON oficial de anotaciones y una carpeta de figuras. Semilla 42.


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from pycocotools.coco import COCO
from skimage.color import rgb2hsv
from tqdm.auto import tqdm


In [ ]:
# semilla fija para muestreos reproducibles
seed = 42
repo_root = Path("../..").resolve()
featured_dir = repo_root / "data" / "featured" / "rocole"
index_path = featured_dir / "index.csv"
coco_json_path = (
    repo_root / "data" / "raw" / "RoCoLe" / "Annotations" / "RoCoLE-coco.json"
)
results_dir = repo_root / "experiments" / "results"
results_dir.mkdir(parents=True, exist_ok=True)

print("repo_root:", repo_root)
print("index_path:", index_path)
print("existe índice:", index_path.exists())
print("coco_json_path:", coco_json_path)
print("existe coco:", coco_json_path.exists())
print("results_dir:", results_dir)
seed


## Cargar el índice partido

El índice es la mesa de trabajo: una fila por foto, con etiqueta, partición y ruta. No rehacemos el split.


In [ ]:
rocole_index = pd.read_csv(index_path)

print("filas:", len(rocole_index))
print("columnas:", list(rocole_index.columns))
rocole_index.head()


## Columnas de identidad que viajan con cada fila

Cada fila debe llevar su identidad: archivo, partición, grupo de enlace y las dos etiquetas. Esas columnas viajan enteras hasta el CSV final.


In [ ]:
# rocole_index: mesa con File, split, link_group y etiquetas

print("filas:", len(rocole_index))
print("splits:", rocole_index["split"].value_counts().to_dict())
id_columns = [
    "File",
    "split",
    "link_group",
    "Binary.Label",
    "Multiclass.Label",
]

print("id_columns:", id_columns)
rocole_index[id_columns].head()


## Cargar las anotaciones COCO

Las máscaras oficiales viven en el JSON COCO de RoCoLe. Lo cargamos una vez para pedir la máscara Leaf por nombre de archivo.


In [ ]:
# si falta: pip install pycocotools

coco = COCO(str(coco_json_path))

print("coco_json_path:", coco_json_path)
print("imágenes en COCO:", len(coco.getImgIds()))
coco


## Tamaños del histograma HSV

El histograma será chico a propósito: ocho cajones de matiz, cuatro de saturación y cuatro de valor. Eso da dieciséis columnas con nombres fijos.


In [ ]:
hue_bins = 8
saturation_bins = 4
value_bins = 4
feature_columns = (
    [f"hsv_h_b{i}" for i in range(hue_bins)]
    + [f"hsv_s_b{i}" for i in range(saturation_bins)]
    + [f"hsv_v_b{i}" for i in range(value_bins)]
)

print("hue_bins:", hue_bins)
print("saturation_bins:", saturation_bins)
print("value_bins:", value_bins)
print("n_features:", len(feature_columns))
feature_columns


## Cargar la máscara COCO por File

En el JSON, file_name es una URL; emparejamos si termina en el File del índice. Devolvemos la unión (OR) de las anotaciones Leaf. Si no hay máscara o no calza, el fallo queda explícito: no caemos a foto completa.


In [ ]:
# coco: anotaciones RoCoLe ya cargadas

def load_coco_leaf_mask(coco, file_name, height, width):
    """Máscara Leaf (OR) emparejando file_name del JSON por sufijo File."""
    matched_images = [
        img
        for img in coco.loadImgs(coco.getImgIds())
        if img["file_name"].endswith(file_name)
    ]
    if len(matched_images) != 1:
        raise ValueError(
            f"esperaba 1 imagen COCO para {file_name}, hay {len(matched_images)}"
        )

    coco_image = matched_images[0]
    # categoría Leaf = 1; imgIds en lista (ids Labelbox son strings)
    ann_ids = coco.getAnnIds(imgIds=[coco_image["id"]], catIds=[1])
    if len(ann_ids) < 1:
        # respaldo por nombre de categoría si catIds=[1] no encuentra
        leaf_cat_ids = [
            cat["id"]
            for cat in coco.loadCats(coco.getCatIds())
            if cat.get("name", "").lower() == "leaf"
        ]
        ann_ids = coco.getAnnIds(imgIds=[coco_image["id"]], catIds=leaf_cat_ids)

    annotations = coco.loadAnns(ann_ids)
    if len(annotations) < 1:
        raise ValueError(f"sin anotaciones Leaf para {file_name}")

    leaf_mask = None
    for annotation in annotations:
        mask_i = coco.annToMask(annotation).astype(bool)
        leaf_mask = mask_i if leaf_mask is None else (leaf_mask | mask_i)

    if leaf_mask.shape != (height, width):
        raise ValueError(
            f"máscara {leaf_mask.shape} vs imagen {(height, width)} para {file_name}"
        )
    return leaf_mask


load_coco_leaf_mask


## Convertir RGB a HSV

Pasamos la foto de RGB a HSV. Esa conversión es solo el cambio de espacio de color; todavía no armamos el histograma.


In [ ]:
def rgb_to_hsv(image):
    """RGB uint8 o float → HSV float con H,S,V en [0, 1]."""
    rgb = np.asarray(image, dtype=np.float64)
    if rgb.max() > 1.0:
        rgb = rgb / 255.0
    hsv = rgb2hsv(rgb)
    return hsv


rgb_to_hsv


## Histograma HSV solo dentro de la máscara

Sobre los píxeles de la máscara contamos el histograma de cada canal, lo normalizamos para que sume uno dentro de la hoja, y devolvemos las dieciséis columnas. Máscara vacía = error explícito.


In [ ]:
# rgb_to_hsv, hue_bins, saturation_bins, value_bins, feature_columns

def extract_hsv_hist_masked(image, leaf_mask):
    """Histograma H/S/V normalizado (suma 1 por canal) solo en la máscara."""
    if not np.any(leaf_mask):
        raise ValueError("máscara Leaf vacía")

    hsv = rgb_to_hsv(image)
    hue_values = hsv[:, :, 0][leaf_mask]
    saturation_values = hsv[:, :, 1][leaf_mask]
    value_values = hsv[:, :, 2][leaf_mask]

    hue_hist, _ = np.histogram(hue_values, bins=hue_bins, range=(0.0, 1.0))
    saturation_hist, _ = np.histogram(
        saturation_values, bins=saturation_bins, range=(0.0, 1.0)
    )
    value_hist, _ = np.histogram(
        value_values, bins=value_bins, range=(0.0, 1.0)
    )

    hue_hist = hue_hist.astype(np.float64)
    saturation_hist = saturation_hist.astype(np.float64)
    value_hist = value_hist.astype(np.float64)

    hue_sum = hue_hist.sum()
    saturation_sum = saturation_hist.sum()
    value_sum = value_hist.sum()
    if hue_sum <= 0 or saturation_sum <= 0 or value_sum <= 0:
        raise ValueError("histograma vacío dentro de la máscara")

    hue_hist = hue_hist / hue_sum
    saturation_hist = saturation_hist / saturation_sum
    value_hist = value_hist / value_sum

    values = np.concatenate([hue_hist, saturation_hist, value_hist])
    return {
        column_name: float(values[i])
        for i, column_name in enumerate(feature_columns)
    }


extract_hsv_hist_masked


## Filas de smoke (train)

Antes del barrido largo tomamos unas pocas filas de train, mezclando sanas y afectadas, con la semilla fija. Solo para probar que el pipeline no se cae.


In [ ]:
# rocole_index, seed, id_columns

train_rows = rocole_index.loc[rocole_index["split"] == "train"].reset_index(drop=True)
healthy_train_sample = (
    train_rows.loc[train_rows["Binary.Label"] == "healthy"]
    .sample(n=3, random_state=seed)
    .reset_index(drop=True)
)
unhealthy_train_sample = (
    train_rows.loc[train_rows["Binary.Label"] == "unhealthy"]
    .sample(n=3, random_state=seed)
    .reset_index(drop=True)
)
smoke_train_rows = pd.concat(
    [healthy_train_sample, unhealthy_train_sample],
    ignore_index=True,
)

print("smoke_train_rows:", len(smoke_train_rows))
print(smoke_train_rows["Binary.Label"].value_counts().to_string())
smoke_train_rows[id_columns]


## Smoke: histograma HSV con máscara, fila a fila

Sobre esas pocas filas leemos la foto, pedimos la máscara Leaf y medimos el histograma HSV. Si falla, NaN en features y mensaje en error. Sin fallback a imagen completa.


In [ ]:
# smoke_train_rows, coco, repo_root
# load_coco_leaf_mask, extract_hsv_hist_masked
# feature_columns, id_columns

smoke_feature_rows = []
for _, row in smoke_train_rows.iterrows():
    feature_row = {column_name: row[column_name] for column_name in id_columns}
    feature_row["error"] = None
    for column_name in feature_columns:
        feature_row[column_name] = np.nan

    try:
        photo_path = repo_root / row["photo_path"]
        sample_image = Image.open(photo_path).convert("RGB")
        height, width = sample_image.size[1], sample_image.size[0]
        leaf_mask = load_coco_leaf_mask(coco, row["File"], height, width)
        hsv_summary = extract_hsv_hist_masked(sample_image, leaf_mask)
        feature_row.update(hsv_summary)
    except Exception as exc:
        # fallo de máscara o lectura: NaN en features, mensaje en error
        feature_row["error"] = str(exc)

    smoke_feature_rows.append(feature_row)

print("smoke_feature_rows:", len(smoke_feature_rows))
print("con error:", sum(1 for r in smoke_feature_rows if r["error"]))
smoke_feature_rows


## Tabla smoke y chequeo de columnas / sumas

Pasamos el smoke a tabla y comprobamos que estén las dieciséis columnas pedidas y que, sin error, los números sean finitos y sumen cerca de uno por canal.


In [ ]:
# smoke_feature_rows, feature_columns, id_columns
# hue_bins, saturation_bins, value_bins

smoke_feature_table = pd.DataFrame(smoke_feature_rows)

expected_columns = id_columns + feature_columns + ["error"]
missing_columns = [
    name for name in expected_columns if name not in smoke_feature_table.columns
]
assert missing_columns == [], f"faltan columnas: {missing_columns}"

ok_mask = smoke_feature_table["error"].isna()
finite_ok = True
channel_sums_ok = True
channel_sum_report = []
if ok_mask.any():
    feature_block = smoke_feature_table.loc[ok_mask, feature_columns]
    finite_ok = bool(np.isfinite(feature_block.to_numpy(dtype=float)).all())

    hue_cols = feature_columns[:hue_bins]
    sat_cols = feature_columns[hue_bins : hue_bins + saturation_bins]
    val_cols = feature_columns[hue_bins + saturation_bins :]
    for idx, row in feature_block.iterrows():
        hue_sum = float(row[hue_cols].sum())
        sat_sum = float(row[sat_cols].sum())
        val_sum = float(row[val_cols].sum())
        channel_sum_report.append(
            {
                "row": int(idx),
                "hue_sum": hue_sum,
                "sat_sum": sat_sum,
                "val_sum": val_sum,
            }
        )
        if not (
            np.isclose(hue_sum, 1.0, atol=1e-6)
            and np.isclose(sat_sum, 1.0, atol=1e-6)
            and np.isclose(val_sum, 1.0, atol=1e-6)
        ):
            channel_sums_ok = False

print("filas:", len(smoke_feature_table))
print("columnas esperadas ok:", missing_columns == [])
print("finitos en filas sin error:", finite_ok)
print("sumas por canal ~1:", channel_sums_ok)
print(channel_sum_report)
smoke_feature_table


## Barrido completo del índice (con máscara)

Recorremos todas las filas del índice con el mismo protocolo masked. Si no hay máscara Leaf, queda error y NaN. No caemos a foto completa.


In [ ]:
# rocole_index, coco, repo_root
# load_coco_leaf_mask, extract_hsv_hist_masked
# feature_columns, id_columns

try:
except ImportError:
    def tqdm(iterable, **kwargs):
        return iterable

inventory_feature_rows = []
for _, row in tqdm(
    rocole_index.iterrows(),
    total=len(rocole_index),
    desc="inventario HSV masked",
):
    feature_row = {column_name: row[column_name] for column_name in id_columns}
    feature_row["error"] = None
    for column_name in feature_columns:
        feature_row[column_name] = np.nan

    try:
        photo_path = repo_root / row["photo_path"]
        sample_image = Image.open(photo_path).convert("RGB")
        height, width = sample_image.size[1], sample_image.size[0]
        # sin fallback a full-image: si no hay máscara, Exception → error
        leaf_mask = load_coco_leaf_mask(coco, row["File"], height, width)
        hsv_summary = extract_hsv_hist_masked(sample_image, leaf_mask)
        feature_row.update(hsv_summary)
    except Exception as exc:
        feature_row["error"] = str(exc)

    inventory_feature_rows.append(feature_row)

print("inventory_feature_rows:", len(inventory_feature_rows))
print(
    "con error:",
    sum(1 for r in inventory_feature_rows if r["error"] is not None),
)


## Mesa de features del inventario HSV masked

La lista del barrido se convierte en la mesa hsv_hist_features_masked: identidad, dieciséis bins y error.


In [ ]:
# inventory_feature_rows

hsv_hist_features_masked = pd.DataFrame(inventory_feature_rows)

print("filas:", len(hsv_hist_features_masked))
print("columnas:", list(hsv_hist_features_masked.columns))
hsv_hist_features_masked.head()


## Chequeo de filas del inventario

Comprobamos una fila por foto del índice, que los conteos por split cuadren y cuántas filas trajeron error.


In [ ]:
# hsv_hist_features_masked, rocole_index

inventory_check = {
    "n_index": len(rocole_index),
    "n_features": len(hsv_hist_features_masked),
    "len_match": len(hsv_hist_features_masked) == len(rocole_index),
    "split_counts_index": rocole_index["split"].value_counts().to_dict(),
    "split_counts_features": hsv_hist_features_masked["split"]
    .value_counts()
    .to_dict(),
    "n_errors": int(hsv_hist_features_masked["error"].notna().sum()),
}

assert inventory_check["len_match"], (
    f"filas features {inventory_check['n_features']} "
    f"!= índice {inventory_check['n_index']}"
)

inventory_check


## Guardar el CSV de features HSV masked

Escribimos hsv_hist_features_masked.csv en la carpeta featured de RoCoLe. Aquí termina la extracción; el EDA va en el otro cuaderno.


In [ ]:
# hsv_hist_features_masked, featured_dir

features_csv_path = featured_dir / "hsv_hist_features_masked.csv"
hsv_hist_features_masked.to_csv(features_csv_path, index=False)

print("guardado:", features_csv_path)
print("existe:", features_csv_path.exists())
features_csv_path


## Conclusiones

Quedó un CSV paralelo de dieciséis bins HSV solo dentro de la máscara Leaf; mismas filas y split que el índice; sin foto completa y sin modelo.
